# 📡 Exercise 01 — Fetch OSM Data

**Day 1 · Guided**

Fetch OpenStreetMap amenity data for a district in your city, inspect the GeoDataFrame, extract coordinates, and save locally.

---

### What is OSM?

OpenStreetMap is a free, community-maintained map of the world. No enforced schema — tags are conventions, not standards.

### What is OSMnx?

[OSMnx](https://osmnx.readthedocs.io/) wraps the Overpass API and returns clean GeoDataFrames. Explore data interactively at [overpass-turbo.eu](https://overpass-turbo.eu).

### 1.1 🏘️ Pick your city **and district**

Look up your city in [`docs/cities.md`](../docs/cities.md) and choose a **district / neighbourhood** within it. Fetching a whole city returns too much data and overloads the API.

> Use `"District, City, Country"` — e.g. `"Södermalm, Stockholm, Sweden"`, `"el Poblenou, Barcelona, Spain"`, `"Kreuzberg, Berlin, Germany"`. Verify on [nominatim.openstreetmap.org](https://nominatim.openstreetmap.org/) that the name resolves.

In [ ]:
# TODO: Replace with your district + city + country
PLACE = "Your District, Your City, Country"  # e.g. "Södermalm, Stockholm, Sweden"

print(f"Working with: {PLACE}")

### 1.2 📥 Fetch amenities

We pass a curated list of amenity types (not *all* amenities) to keep the request small.

In [ ]:
import osmnx as ox

AMENITY_TYPES = [
    "restaurant", "cafe", "fast_food", "bar", "pub",
    "pharmacy", "hospital", "clinic", "dentist",
    "school", "university", "library", "kindergarten",
    "bank", "atm", "post_office",
]

print(f"Fetching amenity data for {PLACE}...")
gdf = ox.features_from_place(PLACE, tags={"amenity": AMENITY_TYPES})
gdf = gdf.reset_index()

print(f"Fetched {len(gdf)} features · {len(gdf.columns)} columns")
print(f"Geometry types:\n{gdf.geometry.geom_type.value_counts()}")

#### Troubleshooting

| Error | Fix |
|-------|-----|
| `InsufficientResponseError` | Check spelling — use `"District, City, Country"` format |
| `ConnectionError` / timeout | Wait 30 s, retry |
| 0 features | Verify name on [nominatim.openstreetmap.org](https://nominatim.openstreetmap.org/) |
| Very slow | Pick a smaller district or reduce `AMENITY_TYPES` |

> ⚠️ The Overpass API is a free shared resource. Don't spam requests — save locally and reload.

### 1.3 🔎 Inspect the raw data

In [ ]:
# Look at one feature — all its columns and values
gdf.iloc[0]

In [ ]:
# TODO: Look at a few more features — are they all structured the same way?
# Try: gdf.iloc[1], gdf.iloc[-1]
# What columns are always present? Which ones are mostly NaN?

**Think:** Which columns does every feature have? Why are most columns mostly NaN?

### 1.4 🌐 Extract coordinates → DataFrame

We need plain `lat`/`lon` columns. `geometry.centroid` works for both Points and Polygons.

In [ ]:
import pandas as pd

# Extract lat/lon from geometry centroids (works for Points and Polygons)
gdf["lat"] = gdf.geometry.centroid.y
gdf["lon"] = gdf.geometry.centroid.x

# Convert to regular DataFrame (drop geometry column)
df = pd.DataFrame(gdf.drop(columns=["geometry"]))

# Rename osmid → id for consistency
if "osmid" in df.columns:
    df = df.rename(columns={"osmid": "id"})

# Drop osmnx metadata columns
for col in ["nodes", "ways"]:
    if col in df.columns:
        df = df.drop(columns=[col])

print(f"Shape: {df.shape}  (rows, columns)")
print(f"Columns: {len(df.columns)}")
df.head()

In [ ]:
# OSMnx already expanded tags into separate columns — let's see what we have
meta_cols = {"element_type", "id", "lat", "lon"}
tag_cols = sorted([c for c in df.columns if c not in meta_cols])
print(f"Number of tag columns: {len(tag_cols)}")

# TODO: Print the first 20 tag column names
# Hint: tag_cols[:20]

In [ ]:
# Verify key columns are present
key_cols = ["id", "lat", "lon", "amenity", "name", "cuisine", "opening_hours"]
for col in key_cols:
    present = col in df.columns
    filled = df[col].notna().sum() if present else 0
    print(f"  {col}: {'✓' if present else '✗'}  ({filled} non-null)")

print(f"\nTotal shape: {df.shape}")

### 1.5 💾 Save locally

In [ ]:
import os
os.makedirs("../data", exist_ok=True)

# Derive a slug from the place name for file naming
city_slug = PLACE.split(",")[0].strip().lower().replace(" ", "_")
gdf.to_file(f"../data/raw_{city_slug}.gpkg", driver="GPKG")

print(f"Saved to ../data/raw_{city_slug}.gpkg")
print(f"Size: {os.path.getsize(f'../data/raw_{city_slug}.gpkg') / 1024:.0f} KB")

### 1.6 👀 First look

In [ ]:
# TODO: Run these commands and study the output

# 1. What are the dtypes?
# df.dtypes

# 2. How much data is missing per column?
# df.info()

# 3. What are the most common amenity types in your city?
# df["amenity"].value_counts().head(10)

---

✅ Raw GeoPackage in `data/` · ✅ Flat DataFrame with lat/lon · ✅ First look at the data

**Next →** [Exercise 02 — Explore & Clean](02_explore_and_clean.ipynb)